## UrbanRide_10 — Gold Funnel Metrics
**Method:** Batch aggregation — Silver → Gold  
**Input:** `urbanride.silver.clickstream` (24.4M events → session-level aggregation)

**Output:** `urbanride.gold.funnel_metrics`  
**Schedule:** Daily · runs after UrbanRide_06 (Silver Clickstream must be complete)  
**Partition:** `session_date`  
**ZORDER:** `ab_test_group`, `customer_id`

**Grain:** one row per `session_id`

**Feature engineering:**
- Session identity: customer, device, city, A/B group, timing
- Funnel progression flags: has_app_open, has_search_ride, has_view_price, has_confirm_booking, has_cancel_booking
- Derived: is_converted, funnel_drop_stage, session_duration_seconds
- Quality flags carried from Silver: is_session_invalid, is_single_event_session

**Answers:**
- Where are users dropping off in the funnel?
- Does the A/B test variant improve conversion rate?
- Which device/city has the best conversion?

**Load type:** Full recompute daily — ensures session_date partitions stay consistent

In [0]:
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    min, max, count, round,
    unix_timestamp
)

CATALOG = 'urbanride'
SILVER  = f'{CATALOG}.silver'
GOLD    = f'{CATALOG}.gold'

spark.sql(f'USE CATALOG {CATALOG}')

# Valid funnel event order
FUNNEL_STAGES = ['app_open', 'search_ride', 'view_price', 'confirm_booking']

print(f'Catalog : {CATALOG}')
print(f'Source  : {SILVER}.clickstream')
print(f'Target  : {GOLD}.funnel_metrics')
print(f'Funnel  : {" → ".join(FUNNEL_STAGES)}')
print('Config ready.')


In [0]:
# Gold funnel_metrics always does a full recompute
# Reason: session-level aggregation must be consistent across all partitions
# A partial incremental update could leave old session_date partitions
# with stale funnel_drop_stage or is_converted values
# 24.4M events → ~11M sessions — full recompute is acceptable daily cost

print('Checking load type...')

table_exists = spark.catalog.tableExists(f'{GOLD}.funnel_metrics')

if not table_exists:
    LOAD_TYPE = 'full'
    print('  Gold table does not exist — FULL LOAD')
else:
    LOAD_TYPE = 'full'
    last_run = spark.sql(
        f'SELECT MAX(_processed_at) FROM {GOLD}.funnel_metrics'
    ).collect()[0][0]
    print(f'  Gold table exists — FULL RECOMPUTE (overwrite)')
    print(f'  Last processed at : {last_run}')

print(f'  Load type : {LOAD_TYPE}')


In [0]:
print('Reading silver.clickstream...')

df_clicks = spark.table(f'{SILVER}.clickstream')

total_events = df_clicks.count()
total_sessions = df_clicks.select('session_id').distinct().count()

print(f'  Total events   : {total_events:,}')
print(f'  Total sessions : {total_sessions:,}')
print()

# Profile event type distribution
print('Event type distribution:')
df_clicks.groupBy('event_type').count().orderBy('count', ascending=False).show()


In [0]:
print('Aggregating events to session level...')

# Aggregate all events per session into one row
# Each funnel stage becomes a boolean flag
# Session timing computed from min/max event_timestamp

df_sessions = df_clicks.groupBy(
    'session_id',
    'customer_id',
    'ab_test_group',
    'ab_test_name',
    'device_type',
    'city',
    'is_session_invalid',
    'is_single_event_session',
).agg(
    # Session timing
    min('event_timestamp').alias('session_start_time'),
    max('event_timestamp').alias('session_end_time'),
    min('event_timestamp').cast('date').alias('session_date'),
    count('event_id').alias('session_event_count'),

    # Funnel progression flags
    # Did this session contain each event type?
    count(when(col('event_type') == 'app_open',        1)).cast('boolean').alias('has_app_open'),
    count(when(col('event_type') == 'search_ride',     1)).cast('boolean').alias('has_search_ride'),
    count(when(col('event_type') == 'view_price',      1)).cast('boolean').alias('has_view_price'),
    count(when(col('event_type') == 'confirm_booking', 1)).cast('boolean').alias('has_confirm_booking'),
    count(when(col('event_type') == 'cancel_booking',  1)).cast('boolean').alias('has_cancel_booking'),
)

print(f'  Sessions aggregated : {df_sessions.count():,}')


In [0]:
print('Deriving conversion and funnel drop stage...')

# Session duration in seconds
df_sessions = df_sessions.withColumn(
    'session_duration_seconds',
    unix_timestamp('session_end_time') - unix_timestamp('session_start_time')
)

# is_converted — did the session result in a confirmed booking?
# This is the primary conversion metric for A/B test analysis
df_sessions = df_sessions.withColumn(
    'is_converted',
    when(col('has_confirm_booking') == True, True).otherwise(False)
)

# funnel_drop_stage — where did the session stop?
# Tells us exactly which stage loses the most users
df_sessions = df_sessions.withColumn(
    'funnel_drop_stage',
    when(col('has_confirm_booking') == True,  'converted')
    .when(col('has_cancel_booking')  == True,  'cancelled')
    .when(col('has_view_price')      == True,  'dropped_at_view_price')
    .when(col('has_search_ride')     == True,  'dropped_at_search')
    .when(col('has_app_open')        == True,  'dropped_at_app_open')
    .otherwise('unknown')
)

converted = df_sessions.filter(col('is_converted') == True).count()
total     = df_sessions.count()
pct = converted / total * 100
print(f'  Total sessions : {total:,}')
print(f'  Converted      : {converted:,} ({pct:.2f}%)')
print()
print('Funnel drop stage distribution:')
df_sessions.groupBy('funnel_drop_stage').count().orderBy('count', ascending=False).show()


In [0]:
df_gold = df_sessions \
    .withColumn('_processed_at', current_timestamp()) \
    .withColumn('_source', lit('silver.clickstream'))

print(f'Final row count   : {df_gold.count():,}')
print(f'Final column count: {len(df_gold.columns)}')
print()
print('Gold schema:')
df_gold.printSchema()


In [0]:
import time
print('Writing gold.funnel_metrics — mode: overwrite...')
t0 = time.time()

df_gold.write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('session_date') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD}.funnel_metrics')

print(f'  Rows written : {df_gold.count():,}')
print(f'  Write time   : {time.time()-t0}s')
print()
print('Running OPTIMIZE + ZORDER...')
spark.sql(f'OPTIMIZE {GOLD}.funnel_metrics ZORDER BY (ab_test_group, customer_id)')
print('  OPTIMIZE + ZORDER complete')


In [0]:
print('=== GOLD FUNNEL METRICS VERIFICATION ===')
print()

df_verify = spark.table(f'{GOLD}.funnel_metrics')
total = df_verify.count()

print(f'  Total rows    : {total:,}')
print(f'  Total columns : {len(df_verify.columns)}')
print()

# Data quality checks
print('Data quality checks:')
checks = [
    ('Null session_id',      df_verify.filter(col('session_id').isNull()).count()),
    ('Null customer_id',     df_verify.filter(col('customer_id').isNull()).count()),
    ('Null session_date',    df_verify.filter(col('session_date').isNull()).count()),
    ('Null ab_test_group',   df_verify.filter(col('ab_test_group').isNull()).count()),
    ('Null is_converted',    df_verify.filter(col('is_converted').isNull()).count()),
    ('Null funnel_drop_stage',df_verify.filter(col('funnel_drop_stage').isNull()).count()),
    ('Unknown drop stage',   df_verify.filter(col('funnel_drop_stage') == 'unknown').count()),
]

all_passed = True
for check, result in checks:
    status = 'PASS' if result == 0 else 'WARN'
    if result > 0: all_passed = False
    print(f'  {status}  {check:<30} : {result:,}')

print()
print('Overall conversion rate:')
converted = df_verify.filter(col('is_converted') == True).count()
pct = converted / total * 100
print(f'  Converted sessions : {converted:,} ({pct:.2f}%)')

print()
print('Funnel drop stage distribution:')
df_verify.groupBy('funnel_drop_stage').count().orderBy('count', ascending=False).show()

print('A/B test conversion comparison — KEY METRIC:')
df_verify.groupBy('ab_test_group').agg(
    count('session_id').alias('total_sessions'),
    count(when(col('is_converted') == True, 1)).alias('converted_sessions'),
    round(
        count(when(col('is_converted') == True, 1)) / count('session_id') * 100, 4
    ).alias('conversion_rate_pct')
).orderBy('ab_test_group').show()

print('Conversion rate by device type:')
df_verify.groupBy('device_type').agg(
    count('session_id').alias('total_sessions'),
    round(
        count(when(col('is_converted') == True, 1)) / count('session_id') * 100, 4
    ).alias('conversion_rate_pct')
).orderBy('conversion_rate_pct', ascending=False).show()

print('Conversion rate by city:')
df_verify.groupBy('city').agg(
    count('session_id').alias('total_sessions'),
    round(
        count(when(col('is_converted') == True, 1)) / count('session_id') * 100, 4
    ).alias('conversion_rate_pct')
).orderBy('conversion_rate_pct', ascending=False).show()

print('Session quality summary:')
from pyspark.sql.functions import sum as spark_sum
df_verify.agg(
    spark_sum(when(col('is_session_invalid')      == True, 1).otherwise(0)).alias('invalid_sessions'),
    spark_sum(when(col('is_single_event_session') == True, 1).otherwise(0)).alias('single_event_sessions'),
    round(spark_sum(when(col('is_single_event_session') == True, 1).otherwise(0)) / count('session_id') * 100, 2).alias('bounce_rate_pct'),
).show(truncate=False)

print('Sample rows:')
df_verify.select(
    'session_id', 'customer_id', 'ab_test_group', 'device_type',
    'session_date', 'session_duration_seconds', 'session_event_count',
    'is_converted', 'funnel_drop_stage'
).limit(5).show(truncate=False)

print()
detail = spark.sql(f'DESCRIBE DETAIL {GOLD}.funnel_metrics').collect()[0]
print(f'  numFiles    : {detail["numFiles"]}')
print(f'  sizeInBytes : {detail["sizeInBytes"]:,}')

print()
if all_passed:
    print('All quality checks passed. Gold funnel metrics ready.')
else:
    print('Some checks flagged — review WARN items above.')
print('Gold layer complete. Next: UrbanRide_11 — Churn Prediction Model')
